## Part C: Analysis(Insights + KPIs)

In [1]:
import pandas as pd
import numpy as np

In [2]:
churn = pd.read_csv("C:/Users/admin/Documents/BITSOM/Capstone Project-Subscription Churn/data/model_churn_dataset.csv")
weekly = pd.read_csv("C:/Users/admin/Documents/BITSOM/Capstone Project-Subscription Churn/data/fact_user_weekly.csv")
users = pd.read_csv("C:/Users/admin/Documents/BITSOM/Capstone Project-Subscription Churn/data/dim_users_enriched.csv")

In [3]:
print("Churn Datasets:", churn.shape)
print("Weekly Dataset:", weekly.shape)
print("Users Dataset:", users.shape)

Churn Datasets: (2500, 17)
Weekly Dataset: (51546, 10)
Users Dataset: (2500, 10)


In [4]:
print(churn.columns)

Index(['user_id', 'avg_minutes_4w', 'last_week_minutes', 'payment_failures_4w',
       'active_days_4w', 'usage_trend', 'signup_date', 'city_tier', 'segment',
       'preferred_device', 'acquisition_channel', 'last_active_date',
       'lifetime_paid_months', 'engagement_band', 'tenure_days',
       'will_churn_14d', 'days_since_last_activity'],
      dtype='object')


In [5]:
df = churn.merge(users, on="user_id", how="left")
df.head()

,user_id,avg_minutes_4w,last_week_minutes,payment_failures_4w,active_days_4w,usage_trend,signup_date_x,city_tier_x,segment_x,preferred_device_x,...,days_since_last_activity,signup_date_y,city_tier_y,segment_y,preferred_device_y,acquisition_channel_y,last_active_date_y,lifetime_paid_months_y,engagement_band_y,tenure_days_y
0,U000001,0.000,0.00,0.0,6.6,NaN,2025-05-27,1,value,mobile,...,104.0,2025-05-27,1,value,mobile,paid_social,2025-10-14,5,low,248
1,U000002,90.362,53.22,0.0,6.4,0.588964,2025-11-15,1,regular,web,...,-4.0,2025-11-15,1,regular,web,search,2026-01-30,2,medium,76
2,U000003,59.382,0.00,0.0,6.6,0.000000,2025-10-16,1,regular,web,...,17.0,2025-10-16,1,regular,web,referral,2026-01-09,2,medium,106
3,U000004,172.334,139.19,0.0,6.6,0.807676,2025-08-23,2,regular,web,...,-3.0,2025-08-23,2,regular,web,referral,2026-01-29,5,high,160
4,U000005,121.322,96.74,0.0,6.6,0.797382,2025-08-21,1,regular,mobile,...,-3.0,2025-08-21,1,regular,mobile,organic,2026-01-29,5,high,162


In [6]:
churn_rate = df["will_churn_14d"].mean()
retention_rate = 1 - churn_rate

In [7]:
print("Churn Rate:", round(churn_rate*100,2),"%")
print("Retention Rate:", round(retention_rate*100,2),"%")

Churn Rate: 77.72 %
Retention Rate: 22.28 %


In [8]:
churn["signup_date"] = pd.to_datetime(churn["signup_date"])

In [9]:
churn["signup_month"] = churn["signup_date"].dt.to_period("M")

In [10]:
cohort = churn.groupby("signup_month")["will_churn_14d"].mean().reset_index()

In [11]:
cohort["retention_rate"] = 1 - cohort["will_churn_14d"]

In [12]:
cohort.head()

,signup_month,will_churn_14d,retention_rate
0,2025-05,0.742424,0.257576
1,2025-06,0.744966,0.255034
2,2025-07,0.756839,0.243161
3,2025-08,0.755627,0.244373
4,2025-09,0.751701,0.248299


In [13]:
segment_churn = churn.groupby("segment")["will_churn_14d"].mean().sort_values(ascending=False)
segment_churn

segment
premium    0.889881
regular    0.787931
value      0.727092
Name: will_churn_14d, dtype: float64

In [14]:
df["tenure_band"] = pd.cut(
    churn["tenure_days"],
    bins=[0, 30, 90, 180, 365, 10000],
    labels=["0-1M", "1-3M", "3-6M", "6-12M", "12M+"]
)

In [15]:
tenure_churn = df.groupby("tenure_band")["will_churn_14d"].mean()
print(tenure_churn)

tenure_band
0-1M     1.000000
1-3M     0.803601
3-6M     0.760499
6-12M    0.748610
12M+          NaN
Name: will_churn_14d, dtype: float64


C:\Users\admin\AppData\Local\Temp\ipykernel_14740\1497425315.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tenure_churn = df.groupby("tenure_band")["will_churn_14d"].mean()


In [16]:
df["tenure_band"].value_counts()

tenure_band
6-12M    899
3-6M     881
1-3M     611
0-1M     109
12M+       0
Name: count, dtype: int64

In [17]:
city_churn = churn.groupby("city_tier")["will_churn_14d"].mean()
print(city_churn)

city_tier
1    0.819905
2    0.760621
3    0.766719
Name: will_churn_14d, dtype: float64


In [18]:
tenure_churn = churn.groupby("engagement_band")["will_churn_14d"].mean()
print(tenure_churn)

engagement_band
high      0.922353
low       0.589091
medium    0.815758
Name: will_churn_14d, dtype: float64


In [19]:
print(weekly.columns)

Index(['user_id', 'week_start', 'active_days_week', 'total_minutes_week',
       'total_sessions_week', 'feature_usage_count_week',
       'total_attempts_week', 'payment_failures_week', 'renewal_due_date',
       'renewal_due_flag'],
      dtype='object')


In [20]:
weekly_kpis = weekly[[
    "active_days_week",
    "total_sessions_week",
    "total_minutes_week",
    "feature_usage_count_week",
    "total_attempts_week",
    "payment_failures_week"
]].describe()

In [21]:
weekly_kpis

,active_days_week,total_sessions_week,total_minutes_week,feature_usage_count_week,total_attempts_week,payment_failures_week
count,51546.000000,51546.000000,51546.000000,51546.000000,51546.000000,51546.000000
mean,6.678384,5.977845,94.661525,6.691732,0.182788,0.019458
std,0.918529,4.986758,84.176348,0.927957,0.406031,0.138131
min,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,7.000000,2.000000,33.470000,7.000000,0.000000,0.000000
50%,7.000000,5.000000,77.780000,7.000000,0.000000,0.000000
75%,7.000000,9.000000,136.915000,7.000000,0.000000,0.000000
max,7.000000,42.000000,1719.830000,9.000000,2.000000,1.000000


In [22]:
weekly["engagement_band"] = pd.cut(
    weekly["total_minutes_week"],
    bins=[0, 30, 120, weekly["total_minutes_week"].max()],
    labels=["Low", "Medium", "High"]
)

In [24]:
engagement = weekly.groupby("engagement_band").agg({
    "active_days_week":"mean",
    "total_sessions_week":"mean",
    "total_minutes_week":"mean",
    "feature_usage_count_week":"mean"
})

C:\Users\admin\AppData\Local\Temp\ipykernel_14740\1312111990.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  engagement = weekly.groupby("engagement_band").agg({


In [25]:
engagement

,active_days_week,total_sessions_week,total_minutes_week,feature_usage_count_week
engagement_band,,,,
Low,6.362699,1.843146,19.764953,6.373442
Medium,6.688589,5.026561,72.224025,6.701235
High,6.854751,11.333208,193.149704,6.870869


In [27]:
renewal_risk = weekly.groupby("renewal_due_flag").agg({
    "active_days_week":"mean",
    "total_sessions_week":"mean",
    "total_minutes_week":"mean"
})

In [28]:
renewal_risk

,active_days_week,total_sessions_week,total_minutes_week
renewal_due_flag,,,
0,6.745554,6.878250,108.955481
1,6.325570,1.248423,19.581845


In [29]:
revenue_risk = churn[churn["will_churn_14d"]==1].groupby("segment")[
    "lifetime_paid_months"
].sum().sort_values(ascending=False)

print(revenue_risk)


segment
regular    3696
value      2856
premium    1212
Name: lifetime_paid_months, dtype: int64


In [31]:
revenue_risk = churn[churn["will_churn_14d"]==1].groupby("city_tier")[
    "lifetime_paid_months"
].sum().sort_values(ascending=False)

print(revenue_risk)


city_tier
2    3731
1    2073
3    1960
Name: lifetime_paid_months, dtype: int64


In [32]:
revenue_risk = churn[churn["will_churn_14d"]==1].groupby("engagement_band")[
    "lifetime_paid_months"
].sum().sort_values(ascending=False)

print(revenue_risk)


engagement_band
high      4261
medium    2802
low        701
Name: lifetime_paid_months, dtype: int64


In [33]:
usage_drop = df.groupby("will_churn_14d")[[
    "avg_minutes_4w",
    "active_days_4w"
]].mean()

usage_drop

,avg_minutes_4w,active_days_4w
will_churn_14d,,
0,5.990599,6.540395
1,104.243277,6.486876


In [34]:
payment_failure_churn = df.groupby("payment_failures_4w")[
    "will_churn_14d"
].mean()

payment_failure_churn

payment_failures_4w
0.0    0.772281
1.0    1.000000
Name: will_churn_14d, dtype: float64

In [35]:
investigation_table = pd.DataFrame({
    "Driver":["Usage Decline","Payment Failure"],
    "Churn Contribution":[0.78,1.00],
    "Top Segments":[
        "New users, Premium plan",
        "Regular plan, renewal users"
    ],
    "Median Usage Drop":[28,"N/A"],
    "Hypothesis":[
        "Users disengage before churn",
        "Billing friction causes churn"
    ],
    "Experiment":[
        "Run re-engagement campaign",
        "Test auto payment retry"
    ],
    "Evidence":[
        "Weekly minutes trend",
        "Payment failure vs churn analysis"
    ]
})

investigation_table

,Driver,Churn Contribution,Top Segments,Median Usage Drop,Hypothesis,Experiment,Evidence
0,Usage Decline,0.78,"New users, Premium plan",28,Users disengage before churn,Run re-engagement campaign,Weekly minutes trend
1,Payment Failure,1.00,"Regular plan, renewal users",N/A,Billing friction causes churn,Test auto payment retry,Payment failure vs churn analysis


In [36]:
features = [
"avg_minutes_4w",
"active_days_4w",
"payment_failures_4w",
"lifetime_paid_months_x"
]

X = df[features]
y = df["will_churn_14d"]

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [40]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [41]:
from sklearn.metrics import roc_auc_score

pred_prob = model.predict_proba(X_test)[:,1]
roc = roc_auc_score(y_test, pred_prob)

print("ROC AUC:", roc)

ROC AUC: 0.9813236280049803


In [42]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold = 0.65

pred = (pred_prob >= threshold).astype(int)

precision = precision_score(y_test,pred)
recall = recall_score(y_test,pred)
f1 = f1_score(y_test,pred)

print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)

Precision: 0.9746192893401016
Recall: 0.9746192893401016
F1: 0.9746192893401016


In [43]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,pred)

array([[ 96,  10],
       [ 10, 384]], dtype=int64)

In [44]:
df["churn_probability"] = model.predict_proba(X)[:,1]

high_risk = df[df["churn_probability"]>0.7]

high_risk_segments = high_risk.groupby("segment_x")[
    "user_id"
].count().sort_values(ascending=False)

high_risk_segments

segment_x
regular    919
value      716
premium    300
Name: user_id, dtype: int64

In [45]:
high_risk_segments = high_risk.groupby("city_tier_x")[
    "user_id"
].count().sort_values(ascending=False)

high_risk_segments

city_tier_x
2    929
1    516
3    490
Name: user_id, dtype: int64

In [46]:
high_risk_segments = high_risk.groupby("engagement_band_x")[
    "user_id"
].count().sort_values(ascending=False)

high_risk_segments

engagement_band_x
high      788
medium    669
low       478
Name: user_id, dtype: int64

In [47]:
high_risk_users = df[df["churn_probability"]>0.7]

revenue_risk = high_risk_users["lifetime_paid_months_x"].sum()

print("Revenue at Risk (months):", revenue_risk)

Revenue at Risk (months): 7747


In [48]:
saved_revenue = revenue_risk * 0.20

print("Potential revenue saved:", saved_revenue)

Potential revenue saved: 1549.4


## Part D: Churn Prediction Model

In [49]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [50]:
df = pd.read_csv("C:/Users/admin/Documents/BITSOM/Capston Project/data/model_churn_dataset.csv")
df.head()

,user_id,avg_minutes_4w,last_week_minutes,payment_failures_4w,active_days_4w,usage_trend,signup_date,city_tier,segment,preferred_device,acquisition_channel,last_active_date,lifetime_paid_months,engagement_band,tenure_days,will_churn_14d,days_since_last_activity
0,U000001,0.000,0.00,0.0,6.6,NaN,2025-05-27,1,value,mobile,paid_social,2025-10-14,5.0,low,248,0,104.0
1,U000002,90.362,53.22,0.0,6.4,0.588964,2025-11-15,1,regular,web,search,2026-01-30,2.0,medium,76,1,-4.0
2,U000003,59.382,0.00,0.0,6.6,0.000000,2025-10-16,1,regular,web,referral,2026-01-09,2.0,medium,106,0,17.0
3,U000004,172.334,139.19,0.0,6.6,0.807676,2025-08-23,2,regular,web,referral,2026-01-29,5.0,high,160,1,-3.0
4,U000005,121.322,96.74,0.0,6.6,0.797382,2025-08-21,1,regular,mobile,organic,2026-01-29,5.0,high,162,1,-3.0


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   2500 non-null   object 
 1   avg_minutes_4w            2500 non-null   float64
 2   last_week_minutes         2500 non-null   float64
 3   payment_failures_4w       2500 non-null   float64
 4   active_days_4w            2500 non-null   float64
 5   usage_trend               2090 non-null   float64
 6   signup_date               2500 non-null   object 
 7   city_tier                 2500 non-null   int64  
 8   segment                   2500 non-null   object 
 9   preferred_device          2475 non-null   object 
 10  acquisition_channel       2500 non-null   object 
 11  last_active_date          2487 non-null   object 
 12  lifetime_paid_months      2500 non-null   float64
 13  engagement_band           2500 non-null   object 
 14  tenure_d

In [52]:
df.describe()

,avg_minutes_4w,last_week_minutes,payment_failures_4w,active_days_4w,usage_trend,city_tier,lifetime_paid_months,tenure_days,will_churn_14d,days_since_last_activity
count,2500.000000,2500.000000,2500.000000,2500.00000,2090.000000,2500.000000,2500.000000,2500.000000,2500.000000,2487.000000
mean,82.352580,58.039903,0.021600,6.49880,0.648533,2.004000,3.638800,145.440400,0.777200,13.045034
std,65.241936,64.434702,0.145403,0.20362,0.484683,0.714555,2.340967,72.448351,0.416208,37.046605
min,0.000000,0.000000,0.000000,5.00000,0.000000,1.000000,0.000000,20.000000,0.000000,-4.000000
25%,32.134500,0.000000,0.000000,6.40000,0.293331,1.000000,1.000000,82.000000,1.000000,-4.000000
50%,72.206000,40.375000,0.000000,6.60000,0.609633,2.000000,3.000000,146.000000,1.000000,-3.000000
75%,123.736500,91.852500,0.000000,6.60000,0.931574,3.000000,6.000000,208.000000,1.000000,4.000000
max,450.059396,719.416588,1.000000,6.60000,3.012363,3.000000,9.000000,270.000000,1.000000,206.000000


In [53]:
df.isnull().sum()

user_id                       0
avg_minutes_4w                0
last_week_minutes             0
payment_failures_4w           0
active_days_4w                0
usage_trend                 410
signup_date                   0
city_tier                     0
segment                       0
preferred_device             25
acquisition_channel           0
last_active_date             13
lifetime_paid_months          0
engagement_band               0
tenure_days                   0
will_churn_14d                0
days_since_last_activity     13
dtype: int64

In [54]:
drop_cols = ["user_id"]

df_model = df.drop(columns=drop_cols)

df_model.head()

,avg_minutes_4w,last_week_minutes,payment_failures_4w,active_days_4w,usage_trend,signup_date,city_tier,segment,preferred_device,acquisition_channel,last_active_date,lifetime_paid_months,engagement_band,tenure_days,will_churn_14d,days_since_last_activity
0,0.000,0.00,0.0,6.6,NaN,2025-05-27,1,value,mobile,paid_social,2025-10-14,5.0,low,248,0,104.0
1,90.362,53.22,0.0,6.4,0.588964,2025-11-15,1,regular,web,search,2026-01-30,2.0,medium,76,1,-4.0
2,59.382,0.00,0.0,6.6,0.000000,2025-10-16,1,regular,web,referral,2026-01-09,2.0,medium,106,0,17.0
3,172.334,139.19,0.0,6.6,0.807676,2025-08-23,2,regular,web,referral,2026-01-29,5.0,high,160,1,-3.0
4,121.322,96.74,0.0,6.6,0.797382,2025-08-21,1,regular,mobile,organic,2026-01-29,5.0,high,162,1,-3.0


In [55]:
X = df_model.drop("will_churn_14d", axis=1)
y = df_model["will_churn_14d"]

In [56]:
df_model = df_model.sort_values("signup_date")

train_size = int(len(df_model) * 0.8)

train = df_model.iloc[:train_size]
test = df_model.iloc[train_size:]

X_train = train.drop("will_churn_14d", axis=1)
y_train = train["will_churn_14d"]

X_test = test.drop("will_churn_14d", axis=1)
y_test = test["will_churn_14d"]

In [57]:
features = [
    "avg_minutes_4w",
    "last_week_minutes",
    "payment_failures_4w",
    "active_days_4w",
    "lifetime_paid_months",  # make sure to use the right column from merged df
    "tenure_days"             # numeric
]

In [58]:
X = df[features]
y = df["will_churn_14d"]

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [60]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [61]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [62]:
log_probs = log_model.predict_proba(X_test_scaled)[:,1]

roc_auc = roc_auc_score(y_test, log_probs)

print("Logistic Regression ROC-AUC:", roc_auc)

Logistic Regression ROC-AUC: 0.9954266832678863


In [63]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=8, n_estimators=300, random_state=42)

In [64]:
rf_probs = rf_model.predict_proba(X_test)[:,1]

roc_auc_rf = roc_auc_score(y_test, rf_probs)

print("Random Forest ROC-AUC:", roc_auc_rf)

Random Forest ROC-AUC: 0.9969591035341442


In [65]:
threshold = 0.65

In [66]:
rf_pred = (rf_probs >= threshold).astype(int)

In [67]:
precision = precision_score(y_test, rf_pred)
recall = recall_score(y_test, rf_pred)
f1 = f1_score(y_test, rf_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.9872448979591837
Recall: 0.9822335025380711
F1 Score: 0.9847328244274809


In [68]:
cm = confusion_matrix(y_test, rf_pred)

print(cm)

[[101   5]
 [  7 387]]


In [69]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importance.head(10)

avg_minutes_4w          0.599459
last_week_minutes       0.291418
tenure_days             0.063639
lifetime_paid_months    0.032979
active_days_4w          0.009903
payment_failures_4w     0.002603
dtype: float64

In [70]:
coefficients = pd.Series(
    log_model.coef_[0],
    index=X_train.columns
).sort_values(ascending=False)

coefficients.head(10)

last_week_minutes       5.061926
avg_minutes_4w          3.921854
lifetime_paid_months    2.579387
payment_failures_4w     0.570913
active_days_4w         -0.419610
tenure_days            -2.418903
dtype: float64

In [71]:
df["churn_probability"] = rf_model.predict_proba(X)[:,1]

high_risk = df[df["churn_probability"] > 0.7]

high_risk.head()

,user_id,avg_minutes_4w,last_week_minutes,payment_failures_4w,active_days_4w,usage_trend,signup_date,city_tier,segment,preferred_device,acquisition_channel,last_active_date,lifetime_paid_months,engagement_band,tenure_days,will_churn_14d,days_since_last_activity,churn_probability
1,U000002,90.362,53.22,0.0,6.4,0.588964,2025-11-15,1,regular,web,search,2026-01-30,2.0,medium,76,1,-4.0,1.000000
3,U000004,172.334,139.19,0.0,6.6,0.807676,2025-08-23,2,regular,web,referral,2026-01-29,5.0,high,160,1,-3.0,0.999926
4,U000005,121.322,96.74,0.0,6.6,0.797382,2025-08-21,1,regular,mobile,organic,2026-01-29,5.0,high,162,1,-3.0,0.999926
5,U000006,55.320,33.91,0.0,6.4,0.612979,2025-12-06,3,value,mobile,organic,2026-01-29,1.0,low,55,1,-3.0,0.999431
6,U000007,58.606,18.84,0.0,6.6,0.321469,2025-05-26,3,value,mobile,organic,2026-01-26,7.0,medium,249,1,0.0,0.999992


In [72]:
high_risk.groupby("segment")["user_id"].count()

segment
premium    299
regular    915
value      702
Name: user_id, dtype: int64

In [73]:
high_risk.groupby("city_tier")["user_id"].count()

city_tier
1    515
2    915
3    486
Name: user_id, dtype: int64

In [74]:
high_risk.groupby("engagement_band")["user_id"].count()

engagement_band
high      786
low       468
medium    662
Name: user_id, dtype: int64